# Lesson 10: LangGraph Studio - The Visual Debugger

## Where We Are

Across Lessons 1-9 we built graphs of increasing complexity - conditional edges, loops, tools, memory, human-in-the-loop, time travel, multi-agent teams. To understand what was happening at runtime we relied on `print` statements and reading the message list back. That works, but it does not scale: bigger graphs need a real inspector.

**LangGraph Studio** is that inspector. It is a visual UI that shows your graph, lets you run it interactively, and exposes State at every checkpoint - no print statements required.

---

## The Mental Model

> **Think of Studio as Chrome DevTools for a graph.** The browser shows you the rendered page; DevTools opens the panel underneath where you can inspect the DOM, set breakpoints, replay network calls, and edit values live. Studio does the same thing for a LangGraph application - the graph still runs, but you get an inspector showing every node, every State write, and controls to step / pause / fork.

---

## What You Will See in Studio

| Panel | What It Shows | Maps to Lesson |
|:---|:---|:---|
| Graph view | Live diagram - nodes light up as they execute | Lesson 1 (anatomy) |
| State view | Current State dict, message-by-message | Lesson 4 (state) |
| Thread list | Every conversation, with full checkpoint chain | Lesson 6 (checkpointer) |
| Interrupt panel | Approve / reject / edit at `interrupt()` points | Lesson 7 (HITL) |
| Time-travel | Click any past checkpoint to inspect or fork | Lesson 8 (time travel) |

Everything you wrote code for, Studio gives you a button for.

---

## How Studio Connects to Your Code

Studio is **not** part of the notebook. It is a separate web UI served by the **LangGraph dev server** (`langgraph dev`). The server:

1. Reads a `langgraph.json` config file in your project root
2. Imports the graph object(s) listed in that config
3. Runs them, exposing checkpoints and state over a local API
4. Opens Studio in your browser, pointed at that API

```
  your code (graph.py)  -->  langgraph.json  -->  langgraph dev (server)  -->  Studio UI
```

So to use Studio we need **two files** outside the notebook: the graph definition and the config. This lesson creates both, then shows you how to launch.

---

## Step 1: Install the CLI

The `langgraph-cli` package provides the `langgraph` command (including `langgraph dev`). The `[inmem]` extra adds the in-memory runtime - good enough for local development.

In [ ]:
%pip install -q "langgraph-cli[inmem]" langgraph langchain langchain-openai python-dotenv

---

## Step 2: Write the Graph File

Studio needs the graph as an **importable Python module**, not as cells in a notebook. We reuse the multi-agent team from Lesson 9 - same supervisor, researcher, writer - and write it to `studio_app/graph.py`.

The key requirement: the file must expose a top-level variable (here `app`) that is the compiled graph.

In [ ]:
import os
os.makedirs("studio_app", exist_ok=True)

In [ ]:
%%writefile studio_app/graph.py
from typing import Annotated, Literal, TypedDict
from pydantic import BaseModel, Field
from langchain_core.messages import AnyMessage, HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import create_react_agent

WEATHER_DATA = {
    "new york": "72F, Sunny",
    "london": "58F, Cloudy",
    "tokyo": "68F, Rainy",
}
POPULATION_DATA = {
    "new york": "8.3 million",
    "london": "8.9 million",
    "tokyo": "13.9 million",
}

@tool
def get_weather(city: str) -> str:
    """Get current weather for a city."""
    return WEATHER_DATA.get(city.lower(), f"No data for {city}")

@tool
def get_population(city: str) -> str:
    """Get population of a city."""
    return POPULATION_DATA.get(city.lower(), f"No data for {city}")

class State(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    next: str

class Route(BaseModel):
    next: Literal["researcher", "writer", "FINISH"] = Field(
        description="Who acts next. FINISH only when fully done."
    )

_supervisor_prompt = (
    "You are a supervisor managing workers: researcher, writer. "
    "Researcher uses tools to gather facts. Writer drafts a paragraph from facts. "
    "Respond FINISH only when the user's original request has been fully delivered."
)
_supervisor_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0).with_structured_output(Route)

def supervisor(state: State) -> dict:
    msgs = [SystemMessage(content=_supervisor_prompt)] + state["messages"]
    decision = _supervisor_llm.invoke(msgs)
    return {"next": decision.next}

_researcher_agent = create_react_agent(
    ChatOpenAI(model="gpt-4o-mini", temperature=0),
    tools=[get_weather, get_population],
    prompt="You are a researcher. Gather facts using tools. Report plainly.",
)

def researcher(state: State) -> dict:
    result = _researcher_agent.invoke({"messages": state["messages"]})
    return {"messages": [HumanMessage(content=result["messages"][-1].content, name="researcher")]}

_writer_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.4)

def writer(state: State) -> dict:
    msgs = [
        SystemMessage(content="You are a writer. Use only facts in the conversation. Draft 2-3 sentences."),
    ] + state["messages"]
    result = _writer_llm.invoke(msgs)
    return {"messages": [HumanMessage(content=result.content, name="writer")]}

def _route(state: State) -> str:
    return END if state["next"] == "FINISH" else state["next"]

_workflow = StateGraph(State)
_workflow.add_node("supervisor", supervisor)
_workflow.add_node("researcher", researcher)
_workflow.add_node("writer", writer)
_workflow.add_edge(START, "supervisor")
_workflow.add_conditional_edges(
    "supervisor", _route,
    {"researcher": "researcher", "writer": "writer", END: END},
)
_workflow.add_edge("researcher", "supervisor")
_workflow.add_edge("writer", "supervisor")

app = _workflow.compile()

Note: we **do not** attach a checkpointer here. When the graph runs under `langgraph dev`, the dev server provides its own checkpointer automatically - one of the things Studio handles for you.

---

## Step 3: Write the `langgraph.json` Config

This is the manifest the dev server reads. It declares:

| Field | Purpose |
|:---|:---|
| `dependencies` | Packages to install for the graph - `.` means the current project |
| `graphs` | Mapping of *name -> import path*. The name is what Studio shows in its dropdown |
| `env` | Where to load environment variables from |

In [ ]:
%%writefile langgraph.json
{
  "dependencies": ["."],
  "graphs": {
    "city_brief_team": "./studio_app/graph.py:app"
  },
  "env": ".env"
}

The `"./studio_app/graph.py:app"` syntax means: *open this file, grab the variable named `app`*. That's the compiled graph Studio will load.

---

## Step 4: Launch Studio

Studio runs as a separate process. Open a terminal in the project root (the folder containing `langgraph.json`) and run:

```bash
langgraph dev
```

What happens:

1. The CLI reads `langgraph.json`
2. Imports your graph(s)
3. Starts a local API server (default: `http://127.0.0.1:2024`)
4. Opens Studio in your default browser, pointed at that API

Expected console output:

```
        Welcome to LangGraph!
  - API:        http://127.0.0.1:2024
  - Docs:       http://127.0.0.1:2024/docs
  - Studio UI:  https://smith.langchain.com/studio/?baseUrl=http://127.0.0.1:2024
```

If the browser does not open automatically, click the Studio UI link.

> **Do not run `langgraph dev` from inside this notebook with `!langgraph dev`.** It is a long-running server. Use a real terminal so you can keep working in the notebook.

---

## Step 5: A Tour of the UI

Once Studio loads with `city_brief_team` selected, you will see four main areas.

### 5a. Graph Panel (left)

A rendered diagram of your graph - the same shape we drew with `draw_mermaid_png()` in earlier lessons, but **live**. As you run the graph, nodes highlight in sequence so you can see exactly which path execution took.

### 5b. Input Panel (top right)

A form for the graph's starting State. For our graph it shows a `messages` field. Type something like:

> Write a short brief about Tokyo's weather and population.

Press **Submit**. Watch the graph panel - `supervisor`, `researcher`, `supervisor`, `writer`, `supervisor` light up in order. No print statements involved.

### 5c. State Panel (right)

After execution, click any node in the graph. The right panel shows the **State at that step** - the full `messages` list, the `next` field, metadata. This is the variable inspector.

### 5d. Threads Panel (left sidebar)

Every run creates a **thread** (just like `thread_id` from Lesson 6). The sidebar lists them. Click an older thread to reload the entire conversation. Studio is using a checkpointer behind the scenes - same idea as Lesson 6 - just exposed through a UI.

---

## Step 6: Doing the Lesson 6-8 Tricks Visually

Studio gives you a button for things we did in code:

| What | Where in Studio | Code Equivalent |
|:---|:---|:---|
| Replay a thread | Re-open it from the Threads sidebar | `invoke(..., config={thread_id})` (Lesson 6) |
| Fork from a past step | Click any node in the graph -> **Fork** button | Pass past `checkpoint_id` to `invoke` (Lesson 8) |
| Edit State, then resume | Click a node -> edit JSON -> **Update & Continue** | `app.update_state(...)` (Lesson 8) |
| Approve / reject an interrupt | Inline approval card when graph pauses | `Command(resume=...)` (Lesson 7) |

Try this hands-on:

1. Run the graph with the Tokyo prompt.
2. After it finishes, click the **researcher** node. Studio shows the State *right after* the researcher ran.
3. Click **Fork** (or **Edit & Re-run**). Change the user message to ask about London.
4. Submit. A new branch executes from that exact checkpoint - the old run is still visible in the Threads sidebar.

That is the Lesson 8 time-travel pattern, done with a mouse instead of code.

---

## Step 7: Verify Your Setup Files

Quick sanity check that the files Studio needs exist.

In [ ]:
import os

expected = ["langgraph.json", "studio_app/graph.py", ".env"]
for path in expected:
    status = "FOUND   " if os.path.exists(path) else "MISSING "
    print(f"{status} {path}")

If `.env` is missing, create it in the project root with at least:

```
OPENAI_API_KEY=sk-...
```

`langgraph dev` reads it automatically because of the `"env": ".env"` line in `langgraph.json`.

---

## Step 8: Troubleshooting

| Symptom | Likely Cause | Fix |
|:---|:---|:---|
| `langgraph: command not found` | CLI not installed in the active environment | `pip install "langgraph-cli[inmem]"` |
| Server starts, browser does not open | Network / popup blocker | Open the Studio URL printed in the console manually |
| `ImportError` in console | `graph.py` has a bad import or missing dependency | Add the package to `dependencies` in `langgraph.json` |
| Graph runs but State panel is empty | You compiled the graph at import time with bad args | Make sure `app = workflow.compile()` (no checkpointer needed) is at module top level |
| `OPENAI_API_KEY` not set | `.env` missing or wrong path | Confirm `.env` is in the same folder as `langgraph.json` |

---

## Summary

### What Changed from Lesson 9

| Lessons 1-9 (Code) | Lesson 10 (Studio) |
|:---|:---|
| Inspect runs via `print` statements | Inspect runs via a live graph diagram |
| Read State by indexing `result["messages"]` | Click any node to see State |
| Fork by passing `checkpoint_id` to `invoke` | Click **Fork** in the UI |
| HITL via `interrupt()` + `Command(resume=...)` | Approval card inline in the UI |
| Threads tracked manually with `thread_id` | Threads listed in a sidebar |

Studio does not change what your graph does. It changes **how you debug it**.

---

### Project Layout for Studio

```
your-project/
  .env                       # secrets
  langgraph.json             # tells Studio where the graph is
  studio_app/
    graph.py                 # module exposing `app` (the compiled graph)
```

Run `langgraph dev` from `your-project/`.

---

### API Reference

| Component | Purpose |
|:---|:---|
| `langgraph-cli[inmem]` | The package providing the `langgraph` command and in-memory runtime |
| `langgraph dev` | Starts the local dev server + opens Studio |
| `langgraph.json` | Manifest: declares dependencies, graphs, env file |
| `"<name>": "path/to/file.py:<symbol>"` | Graph registration format |
| Threads sidebar | Visual equivalent of `thread_id` from Lesson 6 |
| Fork button | Visual equivalent of `checkpoint_id` replay from Lesson 8 |

---

### Key Takeaways

> 1. Studio is a **debugger**, not a runtime - it loads the same graph your code would
> 2. The bridge between your code and Studio is `langgraph.json` + an importable `graph.py`
> 3. Every code-level concept from earlier lessons has a UI button in Studio
> 4. The dev server provides a checkpointer automatically - you do not attach one when targeting Studio
> 5. Use Studio early when building anything non-trivial - the time saved on debugging compounds fast

---

## End of the LangGraph Series

Over ten lessons we walked from a one-node graph to a multi-agent team with memory, human approval, time travel, and a visual debugger:

| # | Concept | Mental Model |
|:---|:---|:---|
| 1 | Anatomy of a graph | The blueprint of a workflow |
| 2 | Conditional edges | A fork in the road |
| 3 | Loops | A while-loop with state |
| 4 | LLM nodes | A smart switch operator |
| 5 | Tools | A concierge with reference books |
| 6 | Checkpointer | The concierge keeps a guest log |
| 7 | Human-in-the-loop | A banker waiting for a signature |
| 8 | Time travel | Video game save points |
| 9 | Multi-agent | A consulting team with a supervisor |
| 10 | Studio | Chrome DevTools for the graph |

Same primitives - **nodes**, **edges**, **State**, **checkpointer** - everything else is composition.